In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.mslandcover.models import HRNetSegmentationModel, get_cls_net
from src.mslandcover.config import HRNET_W18_CONFIG

In [ ]:
model = HRNetSegmentationModel(
    config=HRNET_W18_CONFIG,
    img_decoder_head=True,
    aux_simclr_head=True
)
model.load_state_dict(torch.load('./weights/hrnet_w18/hsv_simclr.pth'))
model.eval()

/tmp/ipykernel_2108623/2918594502.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('./weights/hrnet_w18/hsv_simclr.pth'))


<All keys matched successfully>

In [14]:
import h5py

with h5py.File('/scratch/dhester/mslc/pretrain.hdf5', 'r') as f:
    group = f['pretrain']
    keys = list(group.keys())
    sample = group[keys[0]][()]

sample_tensor = torch.from_numpy(sample).unsqueeze(0)

In [20]:

y_pred, _ = model(sample_tensor)
y_pred_numpy = y_pred.detach().numpy().squeeze()

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 270])

In [ ]:
from src.mslandcover.data.transforms import rgb_to_hsv
import matplotlib.pyplot as plt

y_true = rgb_to_hsv(sample_tensor).squeeze()
y_true_numpy = y_true.numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, name, img in zip(
    axes,
    ['RGB', 'HSV (True)', 'HSV (Pred)'],
    [sample.squeeze().permute(1, 2, 0), y_true_numpy, y_pred_numpy]
):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')

plt.show()